# DRL for Multi-Energy Systems (Toy Workflow)

This notebook demonstrates a minimal end-to-end workflow:
- Create a toy multi-energy environment (electric + thermal coupling)
- Train a PPO agent (`stable-baselines3`)
- Evaluate the policy and visualize a few key signals

> The environment is intentionally simple so the workflow runs quickly. Replace the synthetic profiles with real data and higher-fidelity dynamics as a next step.


In [ ]:
# If running in a fresh environment, install dependencies:
# !pip install -r ../requirements.txt

import sys
from pathlib import Path

# Make ../src importable
ROOT = Path('..').resolve()
sys.path.append(str((ROOT / 'src').resolve()))

import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

from drl_energy_env import MultiEnergyToyEnv


In [ ]:
env = Monitor(MultiEnergyToyEnv(seed=0))
obs, _ = env.reset()
obs


In [ ]:
model = PPO(
    'MlpPolicy',
    env,
    n_steps=2048,
    batch_size=64,
    gamma=0.99,
    learning_rate=3e-4,
    verbose=1,
    seed=0,
)

model.learn(total_timesteps=20_000)


In [ ]:
def rollout(policy, seed=1):
    e = Monitor(MultiEnergyToyEnv(seed=seed))
    obs, _ = e.reset()
    done = False

    soc, temp, pv, load, ambient, price = [], [], [], [], [], []
    grid_import, step_cost, comfort_violation = [], [], []
    rewards = []

    while not done:
        action, _ = policy.predict(obs, deterministic=True)
        obs, r, terminated, truncated, info = e.step(action)
        done = terminated or truncated

        soc.append(obs[0])
        temp.append(obs[1])
        pv.append(obs[2])
        load.append(obs[3])
        ambient.append(obs[4])
        price.append(obs[5])

        grid_import.append(info['grid_import_kw'])
        step_cost.append(info['step_cost'])
        comfort_violation.append(info['comfort_violation'])
        rewards.append(r)

    return {
        'soc': np.array(soc),
        'temp': np.array(temp),
        'pv': np.array(pv),
        'load': np.array(load),
        'ambient': np.array(ambient),
        'price': np.array(price),
        'grid_import': np.array(grid_import),
        'step_cost': np.array(step_cost),
        'comfort_violation': np.array(comfort_violation),
        'reward': np.array(rewards),
    }

traj = rollout(model, seed=123)
{
    'total_cost': float(traj['step_cost'].sum()),
    'total_comfort_violation': float(traj['comfort_violation'].sum()),
    'total_return': float(traj['reward'].sum()),
}


In [ ]:
t = np.arange(len(traj['soc']))

fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

axs[0].plot(t, traj['load'], label='load (kW)')
axs[0].plot(t, traj['pv'], label='pv (kW)')
axs[0].plot(t, traj['grid_import'], label='grid import (kW)')
axs[0].set_ylabel('Power (kW)')
axs[0].legend()
axs[0].grid(True, alpha=0.3)

axs[1].plot(t, traj['soc'])
axs[1].set_ylabel('Battery SOC')
axs[1].grid(True, alpha=0.3)

axs[2].plot(t, traj['temp'], label='indoor temp (°C)')
axs[2].plot(t, traj['ambient'], label='ambient (°C)', alpha=0.7)
axs[2].set_ylabel('Temperature (°C)')
axs[2].set_xlabel('Timestep')
axs[2].legend()
axs[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
